<a href="https://colab.research.google.com/github/arman-hossain45/Datathon_Contest/blob/main/final_linear_regression_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Classification code in details

classification pipeline code

more satisfying code

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV,StratifiedKFold
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from xgboost import XGBClassifier

!pip install catboost
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier,StackingClassifier

import warnings
warnings.filterwarnings('ignore')


train_df=pd.read_csv("dataset.csv")
test_df = pd.read_csv("test.csv")
# keep a copy of the test IDs for submission
test_ids = test_df['PID'].copy()
test_df=test_df.drop(columns=['PID'])

train_df.describe().T
train_df.info()
train_df.shape
train_df.isnull().sum()
#এইখানে সবগুলা কলামের null value দেখাচ্ছে না। এখন যেসব কলামে null value আছে তা দেখার জন্য আমরা নিচের কোডটি লিখবো:
null_values = train_df.isnull().sum()
null_values[null_values > 0]


#এইখানে যাদের missing value অনেক, যেসব কলামের ওই কলামগুলো আমরা delete করে দেব।
#একটা কলাম ডিলিট করতে চাইলে এই কোড লিখবো: df.drop("Car_ID", axis=1, inplace=True)
train_df.drop(
    columns=['Misc Feature', 'Pool QC', 'Alley','Mas Vnr Type','Fireplace Qu'],
    inplace=True
)

#Duplicate value আছে কিনা তা আমরা চেক করলাম।

train_df.duplicated().sum()
train_df.drop_duplicates(inplace=True)


# ============================================================
# NEW: Feature Engineering (simple, high-impact features)
# ============================================================





num=train_df.select_dtypes(include=['int64','float64'])
cat = train_df.select_dtypes(include=['object'])

# see correlation matrics in this pipeline
corr=num.corr()
sns.heatmap(corr,cmap='coolwarm',center=0,annot=False,linewidths=.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

# target column er type check kore nichi - classification hole target categorical o hote pare, numeric label o hote pare
TARGET_COL = 'SalePrice'

if TARGET_COL in num.columns:
    target_corr=(train_df.corr(numeric_only=True)[TARGET_COL].sort_values(ascending=False))
    target_corr
else:
    print(f"Target column '{TARGET_COL}' is categorical, skipping numeric correlation with target.")

#ক্যাটেগরিকাল কলাম যদি অনেক থাকে, তাহলে লেবেল এনকোডার দেখার জন্য এই কোডটা দরকার:

for col in cat:
    print(f"\nColumn: {col}")
    print("Unique count:", train_df[col].nunique())
    print("Values:", train_df[col].unique())

plt.figure(figsize=(15,15))
train_df[num.columns].hist(bins=20,figsize=(15,15))
plt.title("Numerical data distribution in details ")
plt.tight_layout()
plt.show()

for col in num.columns:
  plt.figure(figsize=(8,6))
  sns.histplot(train_df[col],kde=True,bins=20)
  plt.title(f"{col}")
  plt.tight_layout()
  plt.show()

plt.figure(figsize=(15,15))
train_df[num.columns].boxplot()
plt.tight_layout()
plt.show()


def handle_outlier(df, col, lower=None, upper=None):
    df = df.copy()

    if lower is None or upper is None:
        # bounds na dile train data theke calculate kora hocche
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        IQR = q3 - q1
        lower = q1 - 1.5 * IQR
        upper = q3 + 1.5 * IQR

    outlier = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Number of outliers in {col}: {len(outlier)}")

    df[col] = df[col].clip(lower=lower, upper=upper)

    return df, lower, upper


x=train_df.drop(columns=[TARGET_COL])
y=train_df[TARGET_COL]

# classification er jonno target ta categorical/label form e niye asha hocche
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

# ============================================================
# FIX: outlier bounds ekhon shudhu x_train diye calculate kora hocche
# (age puro train_df e kora hoto, oita leakage tairi korto)
# ============================================================
for col in ['Lot Frontage','Lot Area','Wood Deck SF']:
    x_train, lower, upper = handle_outlier(x_train, col)
    x_test, _, _ = handle_outlier(x_test, col, lower=lower, upper=upper)
    test_df, _, _ = handle_outlier(test_df, col, lower=lower, upper=upper)


numerical = x.select_dtypes(include=['int64','float64'])
categorical = x.select_dtypes(include=['object'])

# crate the overall pipeline

num_trans = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_trans =Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("onehot",OneHotEncoder(handle_unknown='ignore'))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_trans,numerical.columns),
        ('cat',cat_trans,categorical.columns)

    ]
)


# base model identify
# NEW: jekhane possible sekhane class_weight='balanced' add kora hoyeche (SMOTE er pashapashi extra help kore)
lr=LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced')
Elas=LogisticRegression(penalty='elasticnet',solver='saga',l1_ratio=0.5,C=1/0.1,max_iter=1000,random_state=42,class_weight='balanced')
Sgd=SGDClassifier(loss='log_loss',max_iter=1000,alpha=0.001,random_state=42,eta0=0.01,learning_rate='optimal',class_weight='balanced')
svr=SVC(max_iter=100,gamma=0.1,kernel='rbf',C=0.6,degree=3,probability=True,random_state=42,class_weight='balanced')
Rg=RandomForestClassifier(n_estimators=100,criterion='gini',min_samples_leaf=2,min_samples_split=3,max_features='sqrt',random_state=42,class_weight='balanced')
Gb=GradientBoostingClassifier(n_estimators=100,subsample=1,min_samples_leaf=3,max_features='sqrt',random_state=42)
Ada=AdaBoostClassifier(n_estimators=156,random_state=42)
Dtr=DecisionTreeClassifier(max_depth=3,min_samples_leaf=3,min_samples_split=3,random_state=42,criterion='gini',class_weight='balanced')
Knr=KNeighborsClassifier(n_neighbors=17,weights='distance',metric='minkowski')
XG=XGBClassifier(n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1)
cat_boost=CatBoostClassifier(verbose=0)
lightgbm=LGBMClassifier()


stacking=StackingClassifier(
    estimators=[
         ('lr',lr),
         ('Rg',Rg),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)
    ],
   final_estimator=LogisticRegression(max_iter=1000)
)
# NOTE: stacking e ekhon shudhu shera 5ta base model rakha hoyeche (age shobgula chilo)
# eta training time onek kombe r overfitting o kombe


voting=VotingClassifier(
    estimators=[
         ('lr',lr),
         ('Rg',Rg),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)
    ],
    voting='soft'
)


model_to_train={
    'Logistic Regression' : lr,
    'Elas':Elas,
    "SGDClassifier":Sgd,
    "SVC":svr,
    "RandomForestClassifier":Rg,
    "GradientBoostingClassifier":Gb,
    "AdaBoostClassifier":Ada,
    'DecisionTreeClassifier':Dtr,
    "KNeighborsClassifier":Knr,
    "XGBClassifier":XG,
    "StackingClassifier":stacking,
    "VotingClassifier":voting,
    "CatBoostClassifier":cat_boost,
    "LGBMClassifier":lightgbm
}


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

# ============================================================
# NEW: shudhu ekbar train-test split e evaluate na kore,
# Stratified K-Fold cross validation diye protita model evaluate kora hocche
# eta leaderboard score er sathe beshi match kore
# ============================================================
result=[]
for model_name,model in model_to_train.items():
    pipe=Pipeline(
        [("preprocessor",preprocessor),
        ("smote",SMOTE(random_state=42)),
        ("model",model)
        ]
    )
    cv_scores = cross_val_score(pipe,x_train,y_train,cv=skf,scoring='f1_weighted',n_jobs=-1)

    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)

    result.append({
        "Model": model_name,
        "CV F1 (mean)": cv_scores.mean(),
        "CV F1 (std)": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test,y_pred),
        "Test F1": f1_score(y_test,y_pred,average='weighted'),
        "Test Precision": precision_score(y_test,y_pred,average='weighted'),
        "Test Recall": recall_score(y_test,y_pred,average='weighted')
    })

res = pd.DataFrame(result).sort_values(by="CV F1 (mean)",ascending=False)
res


xg_pipeline= Pipeline(
    [
        ('preprocessor',preprocessor),
        ('smote',SMOTE(random_state=42)),
        ('feature_selection',SelectFromModel(XGBClassifier(n_estimators=200,random_state=42,n_jobs=-1),threshold='median')),
        ('model',XG)
    ]
)
# NOTE: SelectFromModel add kora hoyeche - kom gurutbopurno feature bad diye
# model ke faster o overfitting-proof korar jonno


# Cross validation
cv_result=cross_val_score(xg_pipeline,x_train,y_train,cv=skf,scoring='f1_weighted')

cv_acc=cv_result.mean()
cv_std=cv_result.std()

cv_acc,cv_std,cv_result


# ============================================================
# NEW: GridSearchCV er bodole RandomizedSearchCV - onek kom shomoy lagbe,
# same range theke random combination try kore prায় kachakachi result dey
# ============================================================
param_grid = {
    "model__n_estimators": [ 900, 1000, 1100],
    "model__learning_rate": [0.04, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 2, 3],
    "model__subsample": [0.4, 0.5, 0.8],
    "model__colsample_bytree": [0.5, 0.8, 1.0],
    "model__reg_alpha": [0.1, 0.2, 0.5],
    "model__reg_lambda": [1, 2, 4]
}

grid=RandomizedSearchCV(
    estimator=xg_pipeline,
    param_distributions=param_grid,
    n_iter=40,
    cv=skf,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

grid.fit(x_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

model = grid.best_estimator_

# feature selection thakar karone shudhu selected feature gulor naam ber kora hocche
selected_mask = model.named_steps["feature_selection"].get_support()
feature_names = model.named_steps["preprocessor"].get_feature_names_out()[selected_mask]

importance = model.named_steps["model"].feature_importances_

importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

print(importance.head(30))

preds=model.predict(test_df)
# label encoder diye abar original class label e ferot newa hocche
preds=target_encoder.inverse_transform(preds)

submission=pd.Dataframe(
    {
        'PID':test_ids,
        TARGET_COL:preds
    }
)
submission.to_csv('submission.csv',index=False)

# Regression problem

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV,KFold
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression,ElasticNet,SGDRegressor,Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from xgboost import XGBRegressor

!pip install catboost
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor,StackingRegressor

import warnings
warnings.filterwarnings('ignore')
import inspect
print(inspect.signature(cross_val_score))

model = XGBClassifier()

model.get_params()


train_df=pd.read_csv("dataset.csv")
test_df = pd.read_csv("test.csv")
# keep a copy of the test IDs for submission
test_ids = test_df['PID'].copy()
test_df=test_df.drop(columns=['PID'])

train_df.describe().T
train_df.info()
train_df.shape
train_df.isnull().sum()
#এইখানে সবগুলা কলামের null value দেখাচ্ছে না। এখন যেসব কলামে null value আছে তা দেখার জন্য আমরা নিচের কোডটি লিখবো:
null_values = train_df.isnull().sum()
null_values[null_values > 0]


#এইখানে যাদের missing value অনেক, যেসব কলামের ওই কলামগুলো আমরা delete করে দেব।
#একটা কলাম ডিলিট করতে চাইলে এই কোড লিখবো: df.drop("Car_ID", axis=1, inplace=True)
train_df.drop(
    columns=['Misc Feature', 'Pool QC', 'Alley','Mas Vnr Type','Fireplace Qu'],
    inplace=True
)

#Duplicate value আছে কিনা তা আমরা চেক করলাম।

train_df.duplicated().sum()
train_df.drop_duplicates(inplace=True)


# ============================================================



num=train_df.select_dtypes(include=['int64','float64'])
cat = train_df.select_dtypes(include=['object'])

# see correlation matrics in this pipeline
corr=num.corr()
sns.heatmap(corr,cmap='coolwarm',center=0,annot=False,linewidths=.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

TARGET_COL = 'SalePrice'

target_corr=(train_df.corr(numeric_only=True)[TARGET_COL].sort_values(ascending=False))
target_corr

#ক্যাটেগরিকাল কলাম যদি অনেক থাকে, তাহলে লেবেল এনকোডার দেখার জন্য এই কোডটা দরকার:

for col in cat:
    print(f"\nColumn: {col}")
    print("Unique count:", train_df[col].nunique())
    print("Values:", train_df[col].unique())

plt.figure(figsize=(15,15))
train_df[num.columns].hist(bins=20,figsize=(15,15))
plt.title("Numerical data distribution in details ")
plt.tight_layout()
plt.show()

for col in num.columns:
  plt.figure(figsize=(8,6))
  sns.histplot(train_df[col],kde=True,bins=20)
  plt.title(f"{col}")
  plt.tight_layout()
  plt.show()

plt.figure(figsize=(15,15))
train_df[num.columns].boxplot()
plt.tight_layout()
plt.show()


def handle_outlier(df, col, lower=None, upper=None):
    df = df.copy()

    if lower is None or upper is None:
        # bounds na dile train data theke calculate kora hocche
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        IQR = q3 - q1
        lower = q1 - 1.5 * IQR
        upper = q3 + 1.5 * IQR

    outlier = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Number of outliers in {col}: {len(outlier)}")

    df[col] = df[col].clip(lower=lower, upper=upper)

    return df, lower, upper


x=train_df.drop(columns=[TARGET_COL])
y=train_df[TARGET_COL]


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# ============================================================
# outlier bounds shudhu x_train diye calculate kora hocche
# (puro train_df e korle leakage tairi hoy)
# ============================================================
for col in ['Lot Frontage','Lot Area','Wood Deck SF']:
    x_train, lower, upper = handle_outlier(x_train, col)
    x_test, _, _ = handle_outlier(x_test, col, lower=lower, upper=upper)
    test_df, _, _ = handle_outlier(test_df, col, lower=lower, upper=upper)


numerical = x.select_dtypes(include=['int64','float64'])
categorical = x.select_dtypes(include=['object'])

# crate the overall pipeline

num_trans = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_trans =Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("onehot",OneHotEncoder(handle_unknown='ignore'))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_trans,numerical.columns),
        ('cat',cat_trans,categorical.columns)

    ]
)


# base model identify
lr=LinearRegression()
Elas=ElasticNet(alpha=0.1)
Sgd=SGDRegressor(loss='squared_error',max_iter=1000,alpha=0.001,random_state=42,eta0=0.01)
svr=SVR(max_iter=100,gamma=0.1,kernel='rbf',C=0.6,degree=3)
Rg=RandomForestRegressor(n_estimators=100,criterion='squared_error',min_samples_leaf=2,min_samples_split=3,max_features='sqrt',random_state=42)
Gb=GradientBoostingRegressor(n_estimators=100,loss='absolute_error',subsample=1,min_samples_leaf=3,max_features='sqrt',random_state=42)
Ada=AdaBoostRegressor(n_estimators=156,loss='linear',random_state=42)
Dtr=DecisionTreeRegressor(max_depth=3,min_samples_leaf=3,min_samples_split=3,random_state=42,criterion='absolute_error')
Knr=KNeighborsRegressor(n_neighbors=17,weights='distance',metric='minkowski')
XG=XGBRegressor(n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1)
cat_boost=CatBoostRegressor(verbose=0)
lightgbm=LGBMRegressor()


stacking=StackingRegressor(
    estimators=[
         ('lr',lr),
         ('Rg',Rg),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)
    ],
   final_estimator=Ridge()
)
# NOTE: stacking e shudhu shera 5ta base model rakha hoyeche (shobgula rakhle training time onek beshi lagto)


voting=VotingRegressor(
    estimators=[
         ('lr',lr),
         ('Rg',Rg),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)
    ]
)


model_to_train={
    'Logistic Regression' : lr,
    'Elas':Elas,
    "SGDRegressor":Sgd,
    "SVR":svr,
    "RandomForestRegressor":Rg,
    "GradientBoostingRegressor":Gb,
    "AdaBoostRegressor":Ada,
    'DecisionTreeRegressor':Dtr,
    "KNeighborsRegressor":Knr,
    "XGBRegressor":XG,
    "StackingRegressor":stacking,
    "VotingRegressor":voting,
    "CatBoostRegressor":cat_boost,
    "LGBMRegressor":lightgbm
}


from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

kf = KFold(n_splits=5,shuffle=True,random_state=42)

# ============================================================
# shudhu ekbar train-test split e evaluate na kore,
# K-Fold cross validation diye protita model evaluate kora hocche
# eta leaderboard score er sathe beshi match kore
# ============================================================
result=[]
for model_name,model in model_to_train.items():
    pipe=Pipeline(
        [("preprocessor",preprocessor),
        ("model",model)
        ]
    )
    cv_scores = cross_val_score(pipe,x_train,y_train,cv=kf,scoring='r2',n_jobs=-1)

    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)

    r2 = r2_score(y_test,y_pred)
    mae = mean_absolute_error(y_test,y_pred)
    mse = mean_squared_error(y_test,y_pred)
    rmse = np.sqrt(mse)

    result.append({
        "Model": model_name,
        "CV R2 (mean)": cv_scores.mean(),
        "CV R2 (std)": cv_scores.std(),
        "Test R2 Score": r2,
        "Test MAE": mae,
        "Test MSE": mse,
        "Test RMSE": rmse
    })

res = pd.DataFrame(result).sort_values(by="CV R2 (mean)",ascending=False)
res


xg_pipeline= Pipeline(
    [
        ('preprocessor',preprocessor),
        ('feature_selection',SelectFromModel(XGBRegressor(n_estimators=200,random_state=42,n_jobs=-1),threshold='median')),
        ('model',XG)
    ]
)
# NOTE: SelectFromModel add kora hoyeche - kom gurutbopurno feature bad diye
# model ke faster o overfitting-proof korar jonno


# Cross validation
cv_result=cross_val_score(xg_pipeline,x_train,y_train,cv=kf,scoring='r2')

cv_acc=cv_result.mean()
cv_std=cv_result.std()

cv_acc,cv_std,cv_result


# ============================================================
# GridSearchCV er bodole RandomizedSearchCV - onek kom shomoy lagbe,
# same range theke random combination try kore prায় kachakachi result dey
# ============================================================
param_grid = {
    "model__n_estimators": [ 900, 1000, 1100],
    "model__learning_rate": [0.04, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 2, 3],
    "model__subsample": [0.4, 0.5, 0.8],
    "model__colsample_bytree": [0.5, 0.8, 1.0],
    "model__reg_alpha": [0.1, 0.2, 0.5],
    "model__reg_lambda": [1, 2, 4]
}

grid=RandomizedSearchCV(
    estimator=xg_pipeline,
    param_distributions=param_grid,
    n_iter=40,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

grid.fit(x_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

model = grid.best_estimator_

# feature selection thakar karone shudhu selected feature gulor naam ber kora hocche
selected_mask = model.named_steps["feature_selection"].get_support()
feature_names = model.named_steps["preprocessor"].get_feature_names_out()[selected_mask]

importance = model.named_steps["model"].feature_importances_

importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

print(importance.head(30))

# final test set metrics (grid theke pawa best model diye)
final_pred = model.predict(x_test)
final_r2 = r2_score(y_test,final_pred)
final_mae = mean_absolute_error(y_test,final_pred)
final_mse = mean_squared_error(y_test,final_pred)
final_rmse = np.sqrt(final_mse)

print(f"\nFinal Test R2 Score: {final_r2:.4f}")
print(f"Final Test MAE: {final_mae:.4f}")
print(f"Final Test MSE: {final_mse:.4f}")
print(f"Final Test RMSE: {final_rmse:.4f}")

preds=model.predict(test_df)

submission=pd.Dataframe(
    {
        'PID':test_ids,
        TARGET_COL:preds
    }
)
submission.to_csv('submission.csv',index=False)